In [16]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

# Read the primary contacts file
file_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment\Contacts_Payment\Contacts.csv"
df_file = pd.read_csv(file_path, low_memory=False)

# Parse session date
df_file["session_dt"] = pd.to_datetime(
    df_file["session_dt"].astype(str).str.strip(),
    format='%d-%m-%Y',
    errors='coerce'
)

# Merge with JN mapping
jn_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\JN_mapping.xlsx"
df_jn = pd.read_excel(jn_path)
merged_df = pd.merge(df_file, df_jn, how="left", left_on="ssi", right_on="current_iss")

conversion_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Conversation_Factors.xlsx"
df_conversion = pd.read_excel(conversion_path)

# Ensure correct data types
merged_df["contacts"] = pd.to_numeric(merged_df["contacts"], errors='coerce')
merged_df["session_dt"] = pd.to_datetime(merged_df["session_dt"], errors='coerce', format='%d-%m-%Y')





# Filter data before using it
filtered_data = merged_df[
    (~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
    (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
    (merged_df["JN"] == "PAYMENTS") &
    (merged_df["ssi"].isin([
        "Order processing Related->Online Payment Issues->Amount Debited Order not confirmed",
        "Product Purchase Queries->Payment related Queries->NA"
    ]))
].copy()

# Group contacts by date
contacts_by_date = (
    filtered_data
    .groupby("session_dt", observed=False)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"session_dt": "Date", "contacts": "total_contacts"})
)

# Calculate conversion %
conversion_df = pd.merge(df_conversion, contacts_by_date, on="Date", how="inner")
conversion_df["conversion_pct"] = (conversion_df["Convertion_factor"] / conversion_df["total_contacts"]).round(2)
conversion_df["date_str"] = conversion_df["Date"].dt.strftime('%d-%m-%Y')

conversion_df["date_str"] = conversion_df["Date"].dt.strftime('%d-%m-%Y')
conversion_map = dict(zip(conversion_df["date_str"], conversion_df["conversion_pct"]))

# Additional filtering for transaction summaries
merged_df["live_response_code_flag"] = merged_df["live_response_code"].apply(
    lambda x: 'SUCCESS' if x == 'SUCCESS' else 'Not_SUCCESS'
)

filtered_data = merged_df[
    (merged_df["payment_instrument"] != 'COD') &
    (merged_df["merchant_id"] == 'mp_flipkart')
].copy()

filtered_data["contacts"] = pd.to_numeric(filtered_data["contacts"], errors='coerce')
group_cols = ["year1", "month1", "wk1", "session_dt"]

# Overall transactions
overall_transaction_data = (
    filtered_data
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "overall_transaction_count"})
)

# Successful transactions
successful_transaction = (
    filtered_data[filtered_data["live_response_code_flag"] == "SUCCESS"]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "successful_count"})
)

# Failed transactions
failed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "FAILED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "failed_count"})
)

# ADONC (Success at gateway, not confirmed at merchant)
adonc_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "SUCCESS")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "adonc_count"})
)

# Order confirmed
order_confirmed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "SUCCESS") &
        (filtered_data["merchant_status"] == "CAPTURED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "order_confirmed_count"})
)

# Transaction failed at merchant side
transaction_failed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "SUCCESS") &
        (filtered_data["merchant_status"] == "INVALIDATED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "transaction_failed_count"})
)

# Combine summary
summary_frames = [
    overall_transaction_data.set_index("session_dt")[["overall_transaction_count"]].rename(columns={"overall_transaction_count": "Total_transaction"}),
    successful_transaction.set_index("session_dt")[["successful_count"]].rename(columns={"successful_count": "Successful_transaction"}),
    failed_data.set_index("session_dt")[["failed_count"]].rename(columns={"failed_count": "Failed_transaction"}),
    adonc_data.set_index("session_dt")[["adonc_count"]].rename(columns={"adonc_count": "Adonc_transaction"}),
    order_confirmed_data.set_index("session_dt")[["order_confirmed_count"]].rename(columns={"order_confirmed_count": "Order_confirmed_transaction"}),
    transaction_failed_data.set_index("session_dt")[["transaction_failed_count"]].rename(columns={"transaction_failed_count": "Transaction_failed_transaction"})
]

summary_df = pd.concat(summary_frames, axis=1).fillna(0)
final_summary = summary_df.T
final_summary.columns = pd.to_datetime(final_summary.columns, errors='coerce').strftime('%d-%m-%Y')
final_summary.reset_index(inplace=True)
final_summary.rename(columns={"index": "Transaction_Data"}, inplace=True)

#print(final_summary)

###-------------------------##
##Adonc------##

# Filter data before using it
filtered_data5 = merged_df[
    (~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
    (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
    (merged_df["JN"] == "PAYMENTS") &
    (merged_df["ssi"].isin([
        "Order processing Related->Online Payment Issues->Amount Debited Order not confirmed"
    ]))
].copy()

# Group contacts by date
contacts_by_date = (
    filtered_data
    .groupby("session_dt", observed=False)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"session_dt": "Date", "contacts": "total_contacts"})
)

# Calculate conversion %
conversion_df = pd.merge(df_conversion, contacts_by_date, on="Date", how="inner")
conversion_df["conversion_pct"] = (
    conversion_df["Convertion_factor"] / conversion_df["total_contacts"]
).round(2)
conversion_df["date_str"] = conversion_df["Date"].dt.strftime('%d-%m-%Y')
conversion_map = dict(zip(conversion_df["date_str"], conversion_df["conversion_pct"]))

# Additional filtering for transaction summaries
merged_df["live_response_code_flag"] = merged_df["live_response_code"].apply(
    lambda x: 'SUCCESS' if x == 'SUCCESS' else 'Not_SUCCESS'
)

filtered_data5 = merged_df[
    (merged_df["payment_instrument"] != 'COD') &
    (merged_df["merchant_id"] == 'mp_flipkart')
].copy()

filtered_data5["contacts"] = pd.to_numeric(filtered_data5["contacts"], errors='coerce')
group_cols = ["year1", "month1", "wk1", "session_dt"]

# Overall transactions
overall_transaction_data1 = (
    filtered_data5
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "overall_transaction_count"})
)

# Successful transactions
successful_transaction1 = (
    filtered_data5[filtered_data5["live_response_code_flag"] == "SUCCESS"]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "successful_count"})
)

# Failed transactions
failed_data1 = (
    filtered_data5[
        (filtered_data5["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data5["transaction_status"] == "FAILED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "failed_count"})
)

# ADONC (Success at gateway, not confirmed at merchant)
adonc_data1 = (
    filtered_data5[
        (filtered_data5["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data5["transaction_status"] == "SUCCESS")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "adonc_count"})
)

# Order confirmed
order_confirmed_data1 = (
    filtered_data5[
        (filtered_data5["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data5["transaction_status"] == "SUCCESS") &
        (filtered_data5["merchant_status"] == "CAPTURED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "order_confirmed_count"})
)

# Transaction failed at merchant side
transaction_failed_data1 = (
    filtered_data5[
        (filtered_data5["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data5["transaction_status"] == "SUCCESS") &
        (filtered_data5["merchant_status"] == "INVALIDATED")
    ]
    .groupby(group_cols)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "transaction_failed_count"})
)

# Combine summary
summary_frames = [
    overall_transaction_data1.set_index("session_dt")[["overall_transaction_count"]].rename(columns={"overall_transaction_count": "Total_transaction"}),
    successful_transaction1.set_index("session_dt")[["successful_count"]].rename(columns={"successful_count": "Successful_transaction"}),
    failed_data1.set_index("session_dt")[["failed_count"]].rename(columns={"failed_count": "Failed_transaction"}),
    adonc_data1.set_index("session_dt")[["adonc_count"]].rename(columns={"adonc_count": "Adonc_transaction"}),
    order_confirmed_data1.set_index("session_dt")[["order_confirmed_count"]].rename(columns={"order_confirmed_count": "Order_confirmed_transaction"}),
    transaction_failed_data1.set_index("session_dt")[["transaction_failed_count"]].rename(columns={"transaction_failed_count": "Transaction_failed_transaction"})
]

summary_df = pd.concat(summary_frames, axis=1).fillna(0)
final_summary1 = summary_df.T
final_summary1.columns = pd.to_datetime(final_summary1.columns, errors='coerce').strftime('%d-%m-%Y')
final_summary1.reset_index(inplace=True)
final_summary1.rename(columns={"index": "Transaction_Data"}, inplace=True)

##print(final_summary1)

###-----------------------##
###Payment_Tranaction_Source
filtered_data1= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS")].copy()

transaction_order1 = ["AndroidApp", "iOSApp", "MobileSite", "ucweb", "WEB"]
filtered_data1["transaction_source"] =pd.Categorical(filtered_data1["transaction_source"], categories=transaction_order1, ordered=True)

transaction_source_summary = (filtered_data1.groupby(["transaction_source","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "transaction_source_count"}))
pivot_transaction_source_countact = transaction_source_summary.pivot_table(index="transaction_source",
                                                                     columns="session_dt",
                                                                     values = "transaction_source_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_transaction_source_countact.columns=pd.to_datetime(pivot_transaction_source_countact.columns, errors='coerce')
pivot_transaction_source_countact = pivot_transaction_source_countact.sort_index(axis=1)
pivot_transaction_source_countact.columns = pivot_transaction_source_countact.columns.strftime('%d-%m-%Y')
pivot_transaction_source_countact.reset_index(inplace=True)

for col in pivot_transaction_source_countact.columns:
    if col in conversion_map:
        pivot_transaction_source_countact[col] = pivot_transaction_source_countact[col] *conversion_map[col]


#print(pivot_transaction_source_countact.head(10))


##-------------------------------------##
#pivot_payment_instrument_countact###

filtered_data2= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS")].copy()

transaction_order2 = ["UPI payment","PHONEPE","UPI_COLLECT","UPI_INTENT","FK_UPI","CREDIT","DEBIT","NET","COIN","EMI","EGV / EGV wallet","WALLET",
]
filtered_data1["payment_instrument"] =pd.Categorical(filtered_data1["payment_instrument"], categories=transaction_order2, ordered=True)

payment_instrument_summary1 = (filtered_data1.groupby(["payment_instrument","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_count"}))

pivot_payment_instrument_countact = payment_instrument_summary1.pivot_table(index="payment_instrument",
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_countact.columns=pd.to_datetime(pivot_payment_instrument_countact.columns, errors='coerce')
pivot_payment_instrument_countact = pivot_payment_instrument_countact.sort_index(axis=1)
pivot_payment_instrument_countact.columns = pivot_payment_instrument_countact.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_countact.reset_index(inplace=True)

for col in pivot_payment_instrument_countact.columns:
    if col in conversion_map:
        pivot_payment_instrument_countact[col] = pivot_payment_instrument_countact[col] *conversion_map[col]


#print(pivot_payment_instrument_countact)

##-------------------------------------##
#pivot_payment_Adonc_instrument_countact###

filtered_data6= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Order processing Related->Online Payment Issues->Amount Debited Order not confirmed")].copy()

transaction_order3 = ["PHONEPE","UPI_COLLECT","UPI_INTENT","FK_UPI","CREDIT","DEBIT","NET","COIN","EMI","EGV / EGV wallet","WALLET"]
filtered_data6["payment_instrument"] =pd.Categorical(filtered_data6["payment_instrument"], categories=transaction_order3, ordered=True)

payment_instrument_Adonc_summary = (filtered_data6.groupby(["payment_instrument","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_Adonc_count"}))

pivot_payment_instrument_Adonc_countact = payment_instrument_Adonc_summary.pivot_table(index="payment_instrument",
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_Adonc_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_Adonc_countact.columns=pd.to_datetime(pivot_payment_instrument_Adonc_countact.columns, errors='coerce')
pivot_payment_instrument_Adonc_countact = pivot_payment_instrument_Adonc_countact.sort_index(axis=1)
pivot_payment_instrument_Adonc_countact.columns = pivot_payment_instrument_Adonc_countact.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_Adonc_countact.reset_index(inplace=True)

for col in pivot_payment_instrument_Adonc_countact.columns:
    if col in conversion_map:
        pivot_payment_instrument_Adonc_countact[col] = pivot_payment_instrument_Adonc_countact[col] *conversion_map[col]


#print(pivot_payment_instrument_Adonc_countact.head(10))

##------------overall summary-------------

filtered_data8= merged_df[(~merged_df["marketplace_id"].isin(["HYPERLOCAL", "RCBP"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) 
                         ].copy()


Overall_refund_summary = (filtered_data8.groupby(["session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_overall_count"}))

Overall_refund_summary["metric"] = "Total_Contacts"
pivot_Overall_refund_summary_countact = Overall_refund_summary.pivot_table(index="metric",
                                                                     columns="session_dt",
                                                                     values = "payment_overall_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_Overall_refund_summary_countact.columns=pd.to_datetime(pivot_Overall_refund_summary_countact.columns, errors='coerce')
pivot_Overall_refund_summary_countact = pivot_Overall_refund_summary_countact.sort_index(axis=1)
pivot_Overall_refund_summary_countact.columns = pivot_Overall_refund_summary_countact.columns.strftime('%d-%m-%Y')
pivot_Overall_refund_summary_countact.reset_index(inplace=True)

for col in pivot_Overall_refund_summary_countact.columns:
    if col in conversion_map:
        pivot_Overall_refund_summary_countact[col] = pivot_Overall_refund_summary_countact[col] *conversion_map[col]


output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment_contacts.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    final_summary.to_excel(writer, sheet_name ='Overall_summary_view', index=False)
    final_summary1.to_excel(writer, sheet_name ='payment_reason_Adonc_countact', index=False )
    pivot_transaction_source_countact.to_excel(writer, sheet_name ='Payment_Tranaction_Source_count', index=False )
    pivot_payment_instrument_countact.to_excel(writer, sheet_name = 'pivot_payment_instrument_Adonc_countact', index=False)
    pivot_Overall_refund_summary_countact.to_excel(writer, sheet_name = 'Overall_count', index=False)
   
print("Excel file successfully saved:",output_path)


Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment_contacts.xlsx


In [14]:
# Safely check all known DataFrames for 'session_dt'
for name in list(globals().keys()):
    try:
        obj = globals()[name]
        if isinstance(obj, pd.DataFrame) and "session_dt" in obj.columns:
            print(f"\n{name}['session_dt'] sample:")
            print(obj["session_dt"].head())
            print("Dtype:", obj["session_dt"].dtype)
    except Exception as e:
        print(f"Skipping {name}: {e}")



df_file['session_dt'] sample:
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: session_dt, dtype: datetime64[ns]
Dtype: datetime64[ns]

merged_df['session_dt'] sample:
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
Name: session_dt, dtype: datetime64[ns]
Dtype: datetime64[ns]

filtered_data['session_dt'] sample:
12   NaT
14   NaT
15   NaT
36   NaT
45   NaT
Name: session_dt, dtype: datetime64[ns]
Dtype: datetime64[ns]

overall_transaction_data['session_dt'] sample:
Series([], Name: session_dt, dtype: datetime64[ns])
Dtype: datetime64[ns]

successful_transaction['session_dt'] sample:
Series([], Name: session_dt, dtype: datetime64[ns])
Dtype: datetime64[ns]

failed_data['session_dt'] sample:
Series([], Name: session_dt, dtype: datetime64[ns])
Dtype: datetime64[ns]

adonc_data['session_dt'] sample:
Series([], Name: session_dt, dtype: datetime64[ns])
Dtype: datetime64[ns]

order_confirmed_data['session_dt'] sample:
Series([], Name: session_dt, dtype: datetime64[ns])
Dtype: datetime64[ns]

transactio